# 04 Evaluation

The system with the best development macro-F1 is chosen and written to `models/deployed.json`, the refusal cutoff is chosen on its development predictions, and only then is the test split scored, once.

With `RESCORE = False` the notebook uses the saved test predictions in `results/*_test.jsonl`. Setting it to True scores the majority and linear systems from notebook 02 on the test split again, and the encoder runs too when their checkpoints from notebook 03 are in `models/runs/`; otherwise their saved test predictions are kept.

The test split was scored here once, for the selection above. [Notebook 06](06_convergence.ipynb) later scored it a second time to test linear models fitted to convergence; that experiment did not change the deployed system ([docs/results.md](../docs/results.md#a-second-test-scoring)).

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Keaane/Ikinyamakuru/blob/main/notebooks/04_evaluation.ipynb)

In [1]:
# Setup: run from the repository root, on a laptop or on Colab.
import os
import subprocess
import sys
import warnings
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("/content/Ikinyamakuru").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Keaane/Ikinyamakuru.git", "/content/Ikinyamakuru"], check=True)
    os.chdir("/content/Ikinyamakuru")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path("src").resolve()))

# Show each warning as one short line.
warnings.showwarning = lambda message, category, *args, **kwargs: print(f"{category.__name__}: {message}")

In [2]:
RESCORE = False  # True scores every system on the test split again

In [3]:
import json

import joblib
import numpy as np

from inkuru.data import align_proba, load_split, read_jsonl, records, write_jsonl
from inkuru.labels import LABELS
from inkuru.metrics import choose_threshold, classification_scores, coverage_stats
from inkuru.paths import BASELINES, MAX_LENGTH, MODELS, RESULTS, RUNS

SYSTEMS = ["majority", "word", "char", "char_balanced", "probe", "full", "full_weighted"]


def scored(rows, predictions):
    by_id = {row["id"]: row for row in predictions}
    gold = [row["label"] for row in rows]
    pred = [by_id[row["id"]]["pred"] for row in rows]
    confidence = np.asarray([by_id[row["id"]]["confidence"] for row in rows])
    return gold, pred, confidence

## Choose on development

Macro-F1 is the selection metric because the test file has a different mix of sections from training, and accuracy would reward filing almost everything as politics. The cutoff is the top-section score with the highest accuracy among filed development articles while still filing at least 80 percent of them.

In [4]:
dev = load_split("dev")
dev_view = {}
for name in SYSTEMS:
    gold, pred, confidence = scored(dev, read_jsonl(RESULTS / f"{name}_dev.jsonl"))
    scores = classification_scores(gold, pred)
    scores.pop("per_class")
    dev_view[name] = {"scores": scores, "gold": gold, "pred": pred, "confidence": confidence}
    print(f"{name:14s} development macro-F1 {scores['macro_f1']:.4f}  accuracy {scores['accuracy']:.4f}")

winner = max(SYSTEMS, key=lambda name: dev_view[name]["scores"]["macro_f1"])
threshold = choose_threshold(dev_view[winner]["gold"], dev_view[winner]["pred"], dev_view[winner]["confidence"])
kind = "majority" if winner == "majority" else ("transformer" if (RUNS / winner).exists() else "baseline")
deployed = {
    "system": winner,
    "kind": kind,
    "threshold": threshold["threshold"],
    "max_length": MAX_LENGTH,
    "labels": LABELS,
    "dev_choice": {
        "metric": "macro_f1",
        "value": dev_view[winner]["scores"]["macro_f1"],
        "rule": "highest development macro-F1 among the seven systems",
    },
    "threshold_choice": threshold,
}
MODELS.mkdir(parents=True, exist_ok=True)
(MODELS / "deployed.json").write_text(json.dumps(deployed, indent=2) + "\n")
print(f"\nselected {winner}; cutoff {threshold['threshold']}, files {threshold['coverage']:.1%} of development, "
      f"{threshold['accuracy_kept']:.1%} of those correct")

majority       development macro-F1 0.0274  accuracy 0.2372
word           development macro-F1 0.5448  accuracy 0.7785
char           development macro-F1 0.6219  accuracy 0.7851
char_balanced  development macro-F1 0.6918  accuracy 0.7785
probe          development macro-F1 0.2395  accuracy 0.5085
full           development macro-F1 0.4769  accuracy 0.6999
full_weighted  development macro-F1 0.4881  accuracy 0.6946



selected char_balanced; cutoff 0.5743, files 80.1% of development, 86.3% of those correct


## Score the test split once

In [5]:
if RESCORE:
    test = load_split("test")
    test_text = [row["text"] for row in test]
    train = load_split("train")
    label = json.loads((BASELINES / "majority.json").read_text())["label"]
    prior = sum(row["label"] == label for row in train) / len(train)
    write_jsonl(RESULTS / "majority_test.jsonl", [
        {"id": row["id"], "gold": row["label"], "pred": label, "confidence": prior, "top": [{"label": label, "p": prior}]}
        for row in test
    ])
    for name in ("word", "char", "char_balanced"):
        pipe = joblib.load(BASELINES / f"{name}.joblib")
        write_jsonl(RESULTS / f"{name}_test.jsonl", records(test, align_proba(pipe, test_text)))
        print("scored", name)
    encoders = [name for name in ("probe", "full", "full_weighted") if (RUNS / name).exists()]
    if encoders:
        import torch
        from torch.utils.data import DataLoader
        from transformers import AutoModelForSequenceClassification, AutoTokenizer

        from inkuru.encoder import TextSet, collate, device_of, predict

        device = device_of()
        for name in encoders:
            tokenizer = AutoTokenizer.from_pretrained(RUNS / name)
            model = AutoModelForSequenceClassification.from_pretrained(RUNS / name).to(device)
            loader = DataLoader(TextSet(test_text, [LABELS.index(row["label"]) for row in test]), batch_size=16,
                                shuffle=False, collate_fn=lambda b: collate(tokenizer, b))
            proba, _ = predict(model, loader, device)
            write_jsonl(RESULTS / f"{name}_test.jsonl", records(test, proba))
            del model
            if device.type == "mps":
                torch.mps.empty_cache()
            print("scored", name)
    for name in ("probe", "full", "full_weighted"):
        if name not in encoders:
            print(f"kept the saved {name} test predictions (no checkpoint in models/runs/{name})")
else:
    print("Using the saved test predictions in results/*_test.jsonl.")

Using the saved test predictions in results/*_test.jsonl.


## Test results

In [6]:
test = load_split("test")
systems = {}
for name in SYSTEMS:
    gold, pred, confidence = scored(test, read_jsonl(RESULTS / f"{name}_test.jsonl"))
    full_test = classification_scores(gold, pred)
    systems[name] = {
        "dev": dev_view[name]["scores"],
        "test": {key: full_test[key] for key in ("accuracy", "macro_precision", "macro_recall", "macro_f1", "n")},
        "per_class": full_test["per_class"],
        "filing": coverage_stats(gold, pred, confidence, threshold["threshold"]) if name == winner else None,
    }
(RESULTS / "metrics.json").write_text(json.dumps({"selected": winner, "threshold": threshold, "systems": systems}, indent=2) + "\n")

print(f"{'system':15s}{'dev F1':>8s}{'test acc':>10s}{'macro-P':>9s}{'macro-R':>9s}{'test F1':>9s}")
for name, row in systems.items():
    test_scores = row["test"]
    print(f"{name:15s}{row['dev']['macro_f1']:8.3f}{test_scores['accuracy']:10.3f}{test_scores['macro_precision']:9.3f}"
          f"{test_scores['macro_recall']:9.3f}{test_scores['macro_f1']:9.3f}")
filing = systems[winner]["filing"]
print(f"\nWith the cutoff, {winner} files {filing['coverage']:.1%} of test articles, and {filing['accuracy_kept']:.1%} of those are correct.")

system           dev F1  test acc  macro-P  macro-R  test F1
majority          0.027     0.091    0.006    0.071    0.012
word              0.545     0.690    0.540    0.476    0.428
char              0.622     0.697    0.708    0.500    0.456
char_balanced     0.692     0.734    0.648    0.594    0.583
probe             0.239     0.520    0.252    0.283    0.244
full              0.477     0.665    0.412    0.461    0.392
full_weighted     0.488     0.674    0.549    0.491    0.435

With the cutoff, char_balanced files 73.8% of test articles, and 84.7% of those are correct.
